## **Importing Packages and Modules**

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import sys
from dotenv import load_dotenv

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (roc_auc_score, classification_report,
                              confusion_matrix, roc_curve)
import os
import scipy.stats as stats

## **Appending Paths**

In [13]:
# Merging the path

sys.path.append(os.path.abspath('../../'))
from src.cleaning_utils import parse_mixed_dates , clean_yes_no_column

## **Loading Env**

In [14]:
# 1. Load the .env file
load_dotenv()

True

## **Reading Data**

In [15]:
folder_path = os.getenv("RAW_DATA_FOLDER")
dir_files=os.listdir(folder_path)

# Creating File Paths
files=[]
for file in dir_files:
    files.append(folder_path+"/"+file)

In [16]:
# Sorting Files for Ensuring Idempotency
files=sorted(files)

# Reading Data
billings=pd.read_csv(files[0])
cc_calls=pd.read_csv(files[1])
emails=pd.read_csv(files[2])
renewal_calls=pd.read_csv(files[3])

In [17]:



# ── Colour palette ──────────────────────────────────────────────
BLUE   = '#1E3A5F'
PINK   = '#E91E8C'
PURPLE = '#7B2D8B'
LIGHT  = '#F8F0FF'
GREY   = '#94A3B8'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   LIGHT,
    'axes.edgecolor':   GREY,
    'axes.labelcolor':  BLUE,
    'xtick.color':      BLUE,
    'ytick.color':      BLUE,
    'text.color':       BLUE,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   11,
    'axes.labelsize':   9,
})

print("✅ Libraries loaded")

✅ Libraries loaded


---
## 3. Load Raw Data

In [18]:
# Use the dataframes already loaded in the previous setup cell
billing = billings.copy()
renewal = renewal_calls.copy()

print(f"Billing  shape: {billing.shape}")
print(f"Renewal  shape: {renewal.shape}")
print(f"\nBilling columns:\n{list(billing.columns)}")
print(f"\nRenewal columns:\n{list(renewal.columns)}")

Billing  shape: (122082, 59)
Renewal  shape: (186534, 41)

Billing columns:
['Co_Ref', 'Renewal_Month', 'Connection_Net', 'Connection_Qty', 'Discount_Amount', 'Sustainability_Score', 'Total_Renewal_Score_New', 'Starting_Connection_Net', 'Starting_Connection_Qty', 'Last_Years_Price', 'Last_Years_Date_Paid', 'Auto_Renewal_Score', 'Status_Scores', 'Anchoring_Score', 'Tenure_Scores', 'Proforma_Auto_Renewal', 'Proforma_World_Pay_Token', 'Proforma_Date', 'Current_Anchorings', 'Current_Anchor_List', 'Payment_Timeframe', 'Registration_Date', 'Proforma_Account_Stage', 'Proforma_Audit_Status', 'Current_Auto_Renewal_Flag', 'Current_World_Pay_Token', 'Renewal_Score_At_Release', 'Proforma_Membership_Status', 'Proforma_Approved_Lists', 'Tenure_Years', 'Band', 'Prospect_Renewal_Date', 'Closed_Date', 'Prospect_Status', 'Starting_Net', 'Starting_Vat', 'Starting_Gross', 'Starting_Membership_Net', 'Starting_Package_Net', 'Starting_PQQ_Net', 'Gross', 'Membership_Net', 'Package_Net', 'PQQNet', 'Total_Net_P

In [19]:
# Quick peek
print("=== BILLING (first 3 rows) ===")
display(billing.head(3))
print("\n=== RENEWAL CALLS (first 3 rows) ===")
display(renewal.head(3))


=== BILLING (first 3 rows) ===


,Co_Ref,Renewal_Month,Connection_Net,Connection_Qty,Discount_Amount,Sustainability_Score,Total_Renewal_Score_New,Starting_Connection_Net,Starting_Connection_Qty,Last_Years_Price,...,Connection_Group,Tenure_Group,#_of_Connection,Last_Renewal,Last_Band,Last_Total_Net_Paid,Last_Connections,Anchor_Group,Renewal_Year,DateTime_Out
0,VT6174,01-11-2024,NaN,NaN,NaN,8.0,42.5,NaN,NaN,799.0,...,1,3,1.0,01-11-2023,Band B,664.0,1.0,1,2024,01-11-2024
1,VD3828,01-08-2025,NaN,NaN,NaN,8.0,41.5,NaN,NaN,799.0,...,1,1,1.0,NaN,NaN,NaN,NaN,1,2025,01-08-2025
2,DV8120,01-03-2025,NaN,NaN,NaN,8.0,33.0,NaN,NaN,799.0,...,1,4+,1.0,01-03-2024,Band C1,749.0,1.0,1,2025,01-03-2025



=== RENEWAL CALLS (first 3 rows) ===


,Call_ID,Call_Direction,Co_Ref,Call_Date,Churn_Category,Complaint_Category,Customer_Reaction_Category,Agent_Renewal_Pitch_Category,Customer_Renewal_Response_Category,Agent_Response_Category,...,Customer_Response,Desire_To_Cancel,Discount_Offered,Justification_Category,Reason_For_Renewal_Category,Agent_Response_To_Cancel_Category,Argument_That_Convinced_Customer_to_Stay_Category,Analysed_Call,Call_Number,Call_Year
0,5.950000e+11,Outbound,UB0899,29-01-2025,NaN,NaN,Not Mentioned,Discussion / Introduction / Inquiry,Discount and Offer,Discount and Offer,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,3,2025
1,5.970000e+11,OUT_BOUND,HN5141,26-02-2025,NaN,NaN,NaN,Price and Cost,Agreement,Customer Communication,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,2,2025
2,5.950000e+11,Outbound,BP5009,24-01-2025,NaN,NaN,NaN,Expiration / Due,Agreement,Accreditation and Certification,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,1,2025


---
## 4. Clean Billing Dataset

### 4.1 Overview & Missing Values

In [20]:
# Shape and data types
print(f"Shape: {billing.shape}")
print(f"Unique Co_Ref: {billing['Co_Ref'].nunique()}")
print(f"Duplicated Co_Ref: {billing['Co_Ref'].duplicated().sum()}")

# Missing value summary
miss = billing.isnull().sum().sort_values(ascending=False)
miss_pct = (miss / len(billing) * 100).round(1)
miss_df = pd.DataFrame({'Missing Count': miss, 'Missing %': miss_pct})
display(miss_df[miss_df['Missing Count'] > 0].head(20))


Shape: (122082, 59)
Unique Co_Ref: 47826
Duplicated Co_Ref: 74256


,Missing Count,Missing %
Last_Years_Date_Paid,122082,100.0
Connection_Qty,114045,93.4
Connection_Net,114045,93.4
Starting_Connection_Net,113537,93.0
Starting_Connection_Qty,113537,93.0
Discount_Amount,108531,88.9
Last_Connections,48380,39.6
Last_Total_Net_Paid,48356,39.6
Last_Band,48311,39.6
Last_Renewal,48291,39.6


### 4.2 Why does billing have duplicates?
Each `Co_Ref` can appear across multiple renewal years (2023, 2024, 2025).  
`Co_Ref + Renewal_Year` is already unique — so each row = one customer × one renewal event.


In [21]:
# Confirm Co_Ref + Renewal_Year is unique
print("Duplicates on Co_Ref + Renewal_Year:", billing.duplicated(subset=['Co_Ref','Renewal_Year']).sum())

# Year distribution
print("\nRenewal_Year distribution:")
print(billing['Renewal_Year'].value_counts().sort_index())

# Outcome distribution
print("\nProspect_Outcome:")
print(billing['Prospect_Outcome'].value_counts())


Duplicates on Co_Ref + Renewal_Year: 0

Renewal_Year distribution:
Renewal_Year
2023    33925
2024    34601
2025    36601
2026    16950
2027        2
2050        3
Name: count, dtype: int64

Prospect_Outcome:
Prospect_Outcome
Won        101226
Churned     12668
Open         8188
Name: count, dtype: int64


### 4.3 Cleaning Steps
1. Parse all date columns  
2. Remove impossible year entries (2027, 2050 — data errors)  
3. Drop "Open" outcomes — no known label, can't use in supervised model  
4. Drop columns with >85% missing  
5. For customers appearing in multiple years → **keep the most recent renewal** (latest year wins)  


In [22]:
billing.shape

(122082, 59)

In [23]:
# Step 1 — Parse date columns
date_cols_b = ['Prospect_Renewal_Date', 'Closed_Date', 'Proforma_Date',
               'Registration_Date', 'Last_Renewal']
for c in date_cols_b:
    billing[c] = pd.to_datetime(billing[c], dayfirst=True, errors='coerce')

# Step 2 — Remove impossible years
billing = billing[billing['Renewal_Year'].isin([2023, 2024, 2025, 2026])].copy()
print(f"After year filter: {billing.shape}")

# Step 3 — Keep only Won / Churned / Open
billing = billing[billing['Prospect_Outcome'].isin(['Won', 'Churned', 'Open'])].copy()
print(f"After outcome filter (Won/Churned/Open only): {billing.shape}")

# Step 4 — Drop extremely sparse columns (>85% missing)
thresh = 0.85
sparse_cols = [c for c in billing.columns if billing[c].isnull().mean() > thresh]
print(f"\nDropping {len(sparse_cols)} sparse columns: {sparse_cols}")
billing.drop(columns=sparse_cols, inplace=True)

# --- Additional Cleaning Steps for NaN values and new features ---
num_cols = [
    'Payment_Timeframe',
    'Total_Net_Paid',
    'Last_Total_Net_Paid',
    'Last_Connections',
    'Tenure_Years',
    'Last_Years_Price', 'Proforma_Approved_Lists',
    '#_of_Connection','Renewal_Score_At_Release',
]

for col in num_cols:
    if col in billing.columns: # Check if column exists before filling
        billing[col].fillna(billing[col].median(), inplace=True)
cat_cols = [
    'Proforma_Auto_Renewal',
    'Proforma_World_Pay_Token',
    'Current_Anchor_List',
    'Proforma_Account_Stage',
    'Proforma_Audit_Status',
    'Last_Band', 'Band', 'Tenure_Group',
    'Anchor_Group',
    'Connection_Group',
    'Proforma_Membership_Status'
]

for col in cat_cols:
    if col in billing.columns: # Check if column exists before filling
        billing[col].fillna('Unknown', inplace=True)
if 'Last_Total_Net_Paid' in billing.columns:
    billing['Last_Total_Net_Paid'].fillna(0, inplace=True)
if 'Last_Connections' in billing.columns:
    billing['Last_Connections'].fillna(0, inplace=True)
if 'Last_Renewal' in billing.columns:
    billing['has_last_renewal'] = billing['Last_Renewal'].notnull().astype(int)
if 'Closed_Date' in billing.columns:
    billing['is_closed'] = billing['Closed_Date'].notnull().astype(int)
# --- End of Additional Cleaning Steps ---

# Step 5 — Deduplicate: keep most recent renewal per customer
billing = (billing
           .sort_values('Prospect_Renewal_Date', ascending=False)
           .drop_duplicates(subset='Co_Ref', keep='first')
           .reset_index(drop=True))

print(f"\n✅ Final billing shape: {billing.shape}")
print(f"   Unique Co_Ref: {billing['Co_Ref'].nunique()}")
print(f"   Duplicated Co_Ref: {billing['Co_Ref'].duplicated().sum()}")
print(f"\nOutcome distribution after cleaning:")
print(billing['Prospect_Outcome'].value_counts())

After year filter: (122077, 59)
After outcome filter (Won/Churned/Open only): (122077, 59)

Dropping 6 sparse columns: ['Connection_Net', 'Connection_Qty', 'Discount_Amount', 'Starting_Connection_Net', 'Starting_Connection_Qty', 'Last_Years_Date_Paid']

✅ Final billing shape: (47825, 55)
   Unique Co_Ref: 47825
   Duplicated Co_Ref: 0

Outcome distribution after cleaning:
Prospect_Outcome
Won        27338
Churned    12300
Open        8187
Name: count, dtype: int64


In [25]:
output_folder_path = os.getenv("CLEAN_DATA_FOLDER")
billing.to_csv(output_folder_path+"/billings_clean.csv",index=False)